In [ ]:
#Packages Used
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sympy import *

In [ ]:
# Analysing the Turbine

var("t x_s z_s phi_s phi_t")      
#phi_t tower angle (to vertical)

# Substructure Variables
x_s = Function("x_s")(t)    #Surge
phi_s = Function("phi_s")(t)    #roll
z_s = Function("z_s")(t)    #Heave

phi_t = Function("phi_t")(t)

# Constant Distances
Lx_st =  0 #Horizontal distance from substructure center to tower base
Lz_st =  89.92 + 10 #Vertical distance from substructure center to tower base
L_t = 120 # Tower Length

pi = 3.14159

# Nacelle Positions
x_t = x_s + Lx_st*cos(phi_s) - Lz_st*sin(phi_s) - L_t*sin(phi_t)
z_t = z_s + Lx_st*sin(phi_s) + Lz_st*cos(phi_s) + L_t*cos(phi_t)

# Relative tower angle ->Later Energy Relations
phi_st = phi_s - phi_t

# Compute/define the velocities here
x_s_dot = diff(x_s, t)
z_s_dot = diff(z_s, t)
phi_s_dot = diff(phi_s, t)

x_t_dot = diff(x_t, t)
z_t_dot = diff(z_t, t)
phi_t_dot = diff(phi_t, t) # will not be used for point masses

phi_st_dot = diff(phi_st, t) # velocity will not be used


var("rho_s B_s H_s W_s m_t")
# substructure density, breadth, height, width;

# Turbine Mass
m_Rotor = 1.1e5           # mass of the rotor [kg]
m_Nacelle = 2.4e5         # mass of the nacelle [kg]
m_Tower = 3.47e5           # mass of the tower [kg
m_t = m_Rotor + m_Nacelle + m_Tower  # total mass of the turbine [kg]


# Define the kinetic energy here (T)
m_s = 7.46633e6 #Substructure mass, including ballast
r_s = 4.7 # Substructure Radius, Assumed constant throughout
cross_area_s =  pi*r_s**2 # Cross-sectional area of the substructure
H_s = 120 + 10 # Spar Buoy Length
rho_s = m_s/(cross_area_s*H_s) # Substructure density, assuming uniform


#~~~~~~~~~~~~~~~~~~~~~~~
# Kinetic Energies: Total = Substructure + Turbine 
#~~~~~~~~~~~~~~~~~~~~~~~
J_s = Ixx = 4.229e9              # Roll inertia [kg m^2]
# Total Kinetic Energy = Velocity Contribution + Rotational Contribution
T_s = 1/2*m_s*(x_s_dot**2 + z_s_dot**2) + 1/2*J_s*(phi_s_dot**2)
T_t = 1/2*m_t*(x_t_dot**2 + z_t_dot**2) + 1/2*0*(phi_t_dot**2) # J_t = 0 as it is assumed to be a point mass
T = T_s + T_t #Total kinetic energy = Substructure + Turbine



var("rho_w g kr_st k_h")

#~~~~~~~~~~~~~~~~~~~~~~~
# Potential Energies: Total = Substructure + Turbine 
#~~~~~~~~~~~~~~~~~~~~~~~
rho_w = 1025      # seawater density [kg/m^3]
g = 9.81                   # gravity [m/s^2]

# , rotational spring stiffness

# Heave Stiffness = Linear Spring

#???
kr_st = 1e9 # rotational stiffness of the tower-substructure connection, very high to approximate a rigid connection
k_h = 1e6 # Heave stiffness, very high to approximate a rigid connection

e_gravity_s = m_s*g*z_s #gravity

draft_s = (cross_area_s*H_s*rho_s)/(cross_area_s*rho_w)
KBuoyancy_s = draft_s/2 # Center of Buoyancy, due to constant shape

submerged_volume_s = cross_area_s*draft_s # Submerged volume, taken in neutral position

J_sub = 1/2*pi*r_s**4 # Circle second moment of area
BM_s = J_sub/submerged_volume_s # Distance from the Center of Buoyancy to the Metacenter = I/submerged_volume

KGravity_s = H_s/2 # Center of Gravity, due to uniform weight
GM_s = KBuoyancy_s + BM_s - KGravity_s
kr_s = rho_w*g*submerged_volume_s*GM_s # Nm/rad

#Substructure Potential Energy

k_s_z = rho_w*g*cross_area_s # Approximate hydrostatic restoring stiffness (linearized buoyancy)

# Total Substructure Potential Energy = Gravitational PE + Heave Spring PE + Rotational Spring PE
V_s = e_gravity_s + 1/2*k_h*x_s**2 + 1/2*k_s_z*z_s**2 + 1/2*kr_s*phi_s**2


# Turbine Potential Energy
e_gravity_t = m_t*g*z_t #gravity
V_t = e_gravity_t + 1/2*kr_st*phi_st**2 # Spring Potential Energy using relative angle

# Total Potential Energy
V = V_s + V_t

# Heave Stiffness = Linear Spring

F_wave = Function("F_wave")(t)
M_wave = Function("M_wave")(t)
F_wind = Function("F_wind")(t)

#~~~~~~~~~~~~~~~~~~~~~~~
# External Forces 
#~~~~~~~~~~~~~~~~~~~~~~~
q = [x_s, z_s, phi_s, phi_t] 
Q = [] 
for qi in q: 
    Q_i = ( 
        F_wind*diff(x_t, qi) + 
        F_wave*diff(z_s, qi) + 
        M_wave*diff(phi_s, qi) 
        ) 
    Q.append(Q_i)

#~~~~~~~~~~~~~~~~~~~~~~~
# The Lagrangian (L) and the Euler-Lagrange Equations
#~~~~~~~~~~~~~~~~~~~~~~~
L = T - V
EOM_x_s = diff( diff(L, x_s_dot), t) - diff(L, x_s) - Q[0]
EOM_z_s = diff( diff(L, z_s_dot), t) - diff(L, z_s) - Q[1]
EOM_phi_s = diff( diff(L, phi_s_dot), t) - diff(L, phi_s) - Q[2]
EOM_phi_t = diff( diff(L, phi_t_dot), t) - diff(L, phi_t) - Q[3]

#~~~~~~~~~~~~~~~~~~~~~~~
# Linearisation
#~~~~~~~~~~~~~~~~~~~~~~~

# dictionaries for substitution
var("x_s_0 x_s_epsilon")
tmp1_x_s = symbols("tmp1_x_s")
psi_x_s = Function("psi_x_s")(t) # perturbation function
tmp2_x_s = symbols("tmp2_x_s")

var("z_s_0 z_s_epsilon")
psi_z_s = Function("psi_z_s")(t) # perturbation function
tmp1_z_s = symbols("tmp1_z_s")
tmp2_z_s = symbols("tmp2_z_s")

var("phi_s_0 phi_s_epsilon")
psi_phi_s = Function("psi_phi_s")(t) # perturbation function
tmp1_phi_s = symbols("tmp1_phi_s")
tmp2_phi_s = symbols("tmp2_phi_s")

var("phi_t_0 phi_t_epsilon")
psi_phi_t = Function("psi_phi_t")(t) # perturbation function
tmp1_phi_t = symbols("tmp1_phi_t")
tmp2_phi_t = symbols("tmp2_phi_t")

subs1_dict = {x_s: tmp1_x_s, z_s: tmp1_z_s, phi_s: tmp1_phi_s, phi_t: tmp1_phi_t}
subs2_dict = {tmp1_x_s: x_s_0 + x_s_epsilon*psi_x_s,
              tmp1_z_s: z_s_0 + z_s_epsilon*psi_z_s,
              tmp1_phi_s: phi_s_0 + phi_s_epsilon*psi_phi_s,
              tmp1_phi_t: phi_t_0 + phi_t_epsilon*psi_phi_t}
subs3_dict = {diff(x_s, (t, 2)): tmp2_x_s, x_s: tmp1_x_s,
              diff(z_s, (t, 2)): tmp2_z_s, z_s: tmp1_z_s,
              diff(phi_s, (t, 2)): tmp2_phi_s, phi_s: tmp1_phi_s,
              diff(phi_t, (t, 2)): tmp2_phi_t, phi_t: tmp1_phi_t}
subs4_dict = {tmp2_x_s: diff(x_s_0 + x_s_epsilon*psi_x_s, (t, 2)), tmp1_x_s: x_s_0 + x_s_epsilon*psi_x_s,
              tmp2_z_s: diff(z_s_0 + z_s_epsilon*psi_z_s, (t, 2)), tmp1_z_s: z_s_0 + z_s_epsilon*psi_z_s,
              tmp2_phi_s: diff(phi_s_0 + phi_s_epsilon*psi_phi_s, (t, 2)), tmp1_phi_s: phi_s_0 + phi_s_epsilon*psi_phi_s,
              tmp2_phi_t: diff(phi_t_0 + phi_t_epsilon*psi_phi_t, (t, 2)), tmp1_phi_t: phi_t_0 + phi_t_epsilon*psi_phi_t}
epsilons_dict = {x_s_epsilon: 1, z_s_epsilon: 1, phi_s_epsilon: 1, phi_t_epsilon: 1}


startpos_dict = {x_s_0: 0, z_s_0: 0, phi_s_0: 0, phi_t_0: 0}

# x_s
EOM_psi_x_s = EOM_x_s.evalf(subs=subs1_dict)
EOM_psi_x_s = EOM_psi_x_s.evalf(subs=subs2_dict)
EOM_lin_x_s = series(EOM_psi_x_s, x_s_epsilon, n=2)

EOM_psi2_x_s = EOM_x_s.evalf(subs=subs3_dict)
EOM_psi2_x_s = EOM_psi2_x_s.evalf(subs=subs4_dict)
EOM_lin_x_s = series(EOM_psi2_x_s, x_s_epsilon, n=2)

EOM_lin_x_s = EOM_lin_x_s.removeO().evalf(subs=epsilons_dict)
EOM_lin_x_s_simplified = EOM_lin_x_s.evalf(subs=startpos_dict) # makes symbolic calcs easier
EOM_lin_x_s_iso = solve(EOM_lin_x_s_simplified, diff(psi_x_s, (t, 2)))

x_s_dotdot = EOM_lin_x_s_iso[0].evalf(subs=startpos_dict)

# z_s
EOM_psi_z_s = EOM_z_s.evalf(subs=subs1_dict)
EOM_psi_z_s = EOM_psi_z_s.evalf(subs=subs2_dict)
EOM_lin_z_s = series(EOM_psi_z_s, z_s_epsilon, n=2)

EOM_psi2_z_s = EOM_z_s.evalf(subs=subs3_dict)
EOM_psi2_z_s = EOM_psi2_z_s.evalf(subs=subs4_dict)
EOM_lin_z_s = series(EOM_psi2_z_s, z_s_epsilon, n=2)

EOM_lin_z_s = EOM_lin_z_s.removeO().evalf(subs=epsilons_dict)
EOM_lin_z_s_simplified = EOM_lin_z_s.evalf(subs=startpos_dict) # makes symbolic calcs easier
EOM_lin_z_s_iso = solve(EOM_lin_z_s_simplified, diff(psi_z_s, (t, 2)))

z_s_dotdot = EOM_lin_z_s_iso[0].evalf(subs=startpos_dict)


# phi_s

EOM_psi_phi_s = EOM_phi_s.evalf(subs=subs1_dict)
EOM_psi_phi_s = EOM_psi_phi_s.evalf(subs=subs2_dict)
EOM_lin_phi_s = series(EOM_psi_phi_s, phi_s_epsilon, n=2)

EOM_psi2_phi_s = EOM_phi_s.evalf(subs=subs3_dict)
EOM_psi2_phi_s = EOM_psi2_phi_s.evalf(subs=subs4_dict)
EOM_lin_phi_s = series(EOM_psi2_phi_s, phi_s_epsilon, n=2)

EOM_lin_phi_s = EOM_lin_phi_s.removeO().evalf(subs=epsilons_dict)
EOM_lin_phi_s_simplified = EOM_lin_phi_s.evalf(subs=startpos_dict) # makes symbolic calcs easier
EOM_lin_phi_s_iso = solve(EOM_lin_phi_s_simplified, diff(psi_phi_s, (t, 2)))

phi_s_dotdot = EOM_lin_phi_s_iso[0].evalf(subs=startpos_dict)


# phi_t

EOM_psi_phi_t = EOM_phi_t.evalf(subs=subs1_dict)
EOM_psi_phi_t = EOM_psi_phi_t.evalf(subs=subs2_dict)
EOM_lin_phi_t = series(EOM_psi_phi_t, phi_t_epsilon, n=2)

EOM_psi2_phi_t = EOM_phi_t.evalf(subs=subs3_dict)
EOM_psi2_phi_t = EOM_psi2_phi_t.evalf(subs=subs4_dict)
EOM_lin_phi_t = series(EOM_psi2_phi_t, phi_t_epsilon, n=2)

EOM_lin_phi_t = EOM_lin_phi_t.removeO().evalf(subs=epsilons_dict)
EOM_lin_phi_t_simplified = EOM_lin_phi_t.evalf(subs=startpos_dict) # makes symbolic calcs easier
EOM_lin_phi_t_iso = solve(EOM_lin_phi_t_simplified, diff(psi_phi_t, (t, 2)))

phi_t_dotdot = EOM_lin_phi_t_iso[0].evalf(subs=startpos_dict)

var("acc1 acc2 acc3 acc4 vel1 vel2 vel3 vel4")

dict_values = { Derivative(psi_x_s, (t,2)): acc1,
                Derivative(psi_z_s, (t,2)): acc2,
                Derivative(psi_phi_s, (t,2)): acc3,
                Derivative(psi_phi_t, (t,2)): acc4,
                Derivative(psi_x_s, t): vel1,
                Derivative(psi_z_s, t): vel2,
                Derivative(psi_phi_s, t): vel3,
                Derivative(psi_phi_t, t): vel4}

EOM_1 = EOM_lin_x_s_simplified.evalf(subs=dict_values)
EOM_2 = EOM_lin_z_s_simplified.evalf(subs=dict_values)
EOM_3 = EOM_lin_phi_s_simplified.evalf(subs=dict_values)
EOM_4 = EOM_lin_phi_t_simplified.evalf(subs=dict_values)

MTRX = linear_eq_to_matrix([EOM_1, EOM_2, EOM_3, EOM_4],
                           [acc1, acc2, acc3, acc4])
# Note: The results per line are the same af from for example EOM_lin_phi_t_iso

M = MTRX[0]
F = MTRX[1]

print(M)

print(F)




Matrix([[8163330.00000000, 0, -69644240.0*cos(psi_phi_s(t)), -83640000.0*cos(psi_phi_t(t))], [0, 8163330.00000000, -69644240.0*sin(psi_phi_s(t)), -83640000.0*sin(psi_phi_t(t))], [-69644240.0000000, -69644240.0*psi_phi_s(t), 11187852460.8000, 8357308800.0*psi_phi_s(t)*sin(psi_phi_t(t)) + 8357308800.0*cos(psi_phi_t(t))], [-83640000.0000000, -83640000.0*psi_phi_t(t), 8357308800.0*psi_phi_t(t)*sin(psi_phi_s(t)) + 8357308800.0*cos(psi_phi_s(t)), 10036800000.0000]])
Matrix([[-69644240.0*vel3**2*sin(psi_phi_s(t)) - 83640000.0*vel4**2*sin(psi_phi_t(t)) + F_wind(t) - 1000000.0*psi_x_s(t)], [69644240.0*vel3**2*cos(psi_phi_s(t)) + 83640000.0*vel4**2*cos(psi_phi_t(t)) + F_wave(t) - 697811.455201275*psi_z_s(t) - 80082267.3], [-8357308800.0*vel4**2*psi_phi_s(t)*cos(psi_phi_t(t)) + 8357308800.0*vel4**2*sin(psi_phi_t(t)) - 99.92*F_wind(t) + M_wave(t) + 592399955.552481*psi_phi_s(t) + 1000000000.0*psi_phi_t(t)], [-8357308800.0*vel3**2*psi_phi_t(t)*cos(psi_phi_s(t)) + 8357308800.0*vel3**2*sin(psi_phi_s(

TypeError: 'Symbol' object is not subscriptable